# 01 - Pull the training data from Splunk SOAR

Step 1 of 3. Writes `data/soar-containers-<N>mo-<date>.jsonl.gz`, which
`02-train-model.ipynb` reads (the newest one in `data/`).

**Tokens.** The three `ph-auth-token`s are read from `SOAR1_TOKEN`,
`SOAR2_TOKEN` and `SOAR3_TOKEN` in the repo-root `.env` file, which is
git-ignored. See `.env.example`. Never paste a token into this notebook.

## Containers, three servers, twelve months

Pulls every container created in the last six months from all three production SOAR
servers, flattens the fields worth analysing, and lifts **`Disposition` out of
`custom_fields`** onto its own column for each container.

Verified against the live API on 2026-09-22. Containers by lookback:

| server | 6mo | 12mo | all time | oldest |
|---|---|---|---|---|
| prod-soar | 23,104 | 48,281 | 106,076 | Nov 2022 |
| prod-soar2 | 53,556 | 68,778 | 68,907 | Mar 2023 |
| prod-soar3 | 7,147 | 7,147 | 7,147 | Apr 2026 |

**Twelve months is the useful limit for labelled data.** `Disposition` was only
introduced in Q4 2025: it starts in December 2025 on prod-soar and around October
2025 on prod-soar2, and NOTHING earlier carries it, back to 2022. Pulling 24 months
or all time adds ~58,000 containers with CEF and metadata but zero dispositions.

Two things the API forced, both worth knowing before you change the code:

- **The filter value must be quoted.** `_filter_create_time__gte="2026-03-22T00:00:00Z"`
  returns 200; the same filter without the quotes returns **400**.
- **`Disposition` is not on every container.** In a 200-row sample it was present on 151.
  A missing one means the field was never set, which is different from a container
  dispositioned as nothing, so it is kept as `None` rather than filled with a blank.

The single-container endpoint reference that used to be this notebook is still here, at
the bottom, and now runs against a container from the pull instead of a hardcoded id.

## Configuration

In [ ]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import pandas as pd
import requests
import urllib3
from dotenv import find_dotenv, load_dotenv

# The SOAR boxes use certificates this client does not chain to. VERIFY_TLS is
# left on as the default so turning it off is a decision someone makes here,
# visibly, rather than a line buried in the request helper.
VERIFY_TLS = False
if not VERIFY_TLS:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Tokens come from the .env file at the repo root, which is git-ignored.
# find_dotenv walks up from the working directory until it finds it.
load_dotenv(find_dotenv(usecwd=True))

SERVERS = [
    {"name": "prod-soar",  "base": "https://prod-soar.mdr.tekstream.com",  "token_env": "SOAR1_TOKEN"},
    {"name": "prod-soar2", "base": "https://prod-soar2.mdr.tekstream.com", "token_env": "SOAR2_TOKEN"},
    {"name": "prod-soar3", "base": "https://prod-soar3.mdr.tekstream.com", "token_env": "SOAR3_TOKEN"},
]
_missing = [s["token_env"] for s in SERVERS if not os.environ.get(s["token_env"])]
assert not _missing, f"set {', '.join(_missing)} in .env (see .env.example)"
for s in SERVERS:
    s["token"] = os.environ[s["token_env"]]

MONTHS_BACK = 12
# Which timestamp the window applies to. create_time is when the container was
# ingested; switch to container_update_time to catch older containers that were
# touched inside the window instead.
TIME_FIELD = "create_time"

PAGE_SIZE = 1000          # 1000 returns in 1-2s per page; 84 pages for the full pull
REQUEST_TIMEOUT = 90
MAX_RETRIES = 4

OUTPUT_DIR = "data"   # git-ignored; 02-train-model.ipynb reads from here

# The custom field asked for by name. Kept as a constant because it is a string
# key in someone else's schema: if it is renamed in SOAR, this is the one place
# that has to change.
DISPOSITION_FIELD = "Disposition"


def months_ago(n: int, now: datetime | None = None) -> datetime:
    """`n` calendar months back, clamped to the end of a shorter month."""
    now = now or datetime.now(timezone.utc)
    month = now.month - n
    year = now.year + (month - 1) // 12
    month = (month - 1) % 12 + 1
    day = min(now.day, [31, 29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                        31, 30, 31, 30, 31, 31, 30, 31, 30, 31][month - 1])
    return now.replace(year=year, month=month, day=day)


SINCE = months_ago(MONTHS_BACK)
SINCE_ISO = SINCE.strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"window: {TIME_FIELD} >= {SINCE_ISO}  ({MONTHS_BACK} months back from {datetime.now(timezone.utc):%Y-%m-%d})")

# --- CEF ---------------------------------------------------------------
# CEF lives on ARTIFACTS, not on containers, so it is pulled from /rest/artifact
# in bulk and grouped back by container id. 257,616 artifacts in this window
# across the three servers, averaging ~2.7KB of CEF each: roughly 700MB of JSON,
# which is why only `container` and `cef` are kept per artifact and why the CSV
# is gzipped when CEF is included.
FETCH_CEF = True
ARTIFACT_PAGE_SIZE = 1000

# --- output ------------------------------------------------------------
# JSONL is the primary format because CEF is NESTED. A CSV can only hold it
# as a JSON string inside a field, so every reader has to json.loads a column
# to get back what was already structured, and a stray comma or newline in a
# CEF value is one quoting bug away from corrupting the row. JSONL keeps the
# list a list and streams line by line instead of loading 700MB at once.
#
# The CSV stays available for anything that wants a flat table, and drops the
# CEF column rather than pretending a spreadsheet can carry it.
WRITE_JSONL = True
WRITE_CSV = True
CSV_INCLUDES_CEF = False
WRITE_PARQUET = True

## The pull

Paged, and **sorted by `id` ascending** rather than by time. A deep pull takes minutes and
new containers arrive while it runs; sorted newest-first every insert shifts the whole
window and rows get duplicated across page boundaries or skipped entirely. Ascending ids
are stable, so anything that arrives mid-pull lands after the pages already read.

`(server, id)` is deduplicated anyway, because trusting that argument without checking it
is how the bug comes back quietly.

In [ ]:
def fetch_page(server: dict, page: int) -> dict:
    """One page of containers, with retries on transient failures."""
    params = {
        # The quotes are REQUIRED: without them the API answers 400.
        f"_filter_{TIME_FIELD}__gte": f'"{SINCE_ISO}"',
        "page_size": PAGE_SIZE,
        "page": page,
        "sort": "id",
        "order": "asc",
    }
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(
                f"{server['base']}/rest/container",
                headers={"ph-auth-token": server["token"]},
                params=params,
                verify=VERIFY_TLS,
                timeout=REQUEST_TIMEOUT,
            )
            r.raise_for_status()
            return r.json()
        except Exception as exc:  # noqa: BLE001 - retry anything transient, surface the rest
            last_error = exc
            if attempt == MAX_RETRIES - 1:
                break
            time.sleep(2 ** attempt)
    raise RuntimeError(f"{server['name']} page {page} failed after {MAX_RETRIES} tries: {last_error}")


def fetch_server(server: dict) -> list[dict]:
    """Every container in the window from one server."""
    first = fetch_page(server, 0)
    total = first.get("count", 0)
    rows = list(first.get("data", []))
    pages = -(-total // PAGE_SIZE)  # ceiling
    print(f"{server['name']}: {total:,} containers, {pages} pages", flush=True)

    for page in range(1, pages):
        batch = fetch_page(server, page).get("data", [])
        if not batch:
            print(f"  page {page} came back empty, stopping early", flush=True)
            break
        rows.extend(batch)
        if page % 10 == 0 or page == pages - 1:
            print(f"  {len(rows):,}/{total:,}", flush=True)

    for row in rows:
        row["_server"] = server["name"]
    return rows


started = time.time()
raw = []
for srv in SERVERS:
    raw.extend(fetch_server(srv))
print(f"\n{len(raw):,} rows from {len(SERVERS)} servers in {time.time() - started:.0f}s")

## Flatten, with `Disposition` lifted out

`custom_fields` is kept whole in its own column as well, so a field nobody has asked for
yet is still in the data rather than thrown away at pull time.

In [ ]:
KEEP = [
    "id", "name", "label", "status", "severity", "sensitivity", "container_type",
    "owner_name", "tenant", "create_time", "start_time", "end_time", "close_time",
    "due_time", "open_time", "container_update_time", "artifact_count", "in_case",
    "current_phase", "workflow_name", "external_id", "source_data_identifier", "tags",
]


def flatten(row: dict) -> dict:
    custom = row.get("custom_fields") or {}
    out = {"server": row.get("_server")}
    out.update({k: row.get(k) for k in KEEP})
    # The ask: Disposition, per container, out of custom_fields.
    # Absent means never set, which is not the same as empty, so it stays None.
    out["disposition"] = custom.get(DISPOSITION_FIELD)
    out["custom_fields"] = custom
    return out


containers = pd.DataFrame([flatten(r) for r in raw])

before = len(containers)
containers = containers.drop_duplicates(subset=["server", "id"], keep="last")
if before != len(containers):
    print(f"dropped {before - len(containers):,} duplicate (server, id) rows")

for col in ("create_time", "close_time", "container_update_time"):
    containers[col] = pd.to_datetime(containers[col], errors="coerce", utc=True)

containers = containers.sort_values(["server", "create_time"]).reset_index(drop=True)
print(f"{len(containers):,} containers")
containers.head()

## What came back

In [ ]:
print("per server\n")
summary = containers.groupby("server").agg(
    containers=("id", "size"),
    with_disposition=("disposition", "count"),
    earliest=("create_time", "min"),
    latest=("create_time", "max"),
)
summary["disposition_%"] = (100 * summary["with_disposition"] / summary["containers"]).round(1)
display(summary)

print("\ndisposition values across all three servers\n")
counts = containers["disposition"].value_counts(dropna=False).to_frame("containers")
counts["%"] = (100 * counts["containers"] / len(containers)).round(1)
display(counts)

missing = containers["disposition"].isna().sum()
print(f"\n{missing:,} of {len(containers):,} containers ({100 * missing / len(containers):.1f}%) have no "
      f"{DISPOSITION_FIELD} set at all.")

In [ ]:
# Every custom field seen, not just Disposition: useful for spotting a rename
# or a second field that means the same thing on one server and not the others.
from collections import Counter

seen = Counter()
for fields in containers["custom_fields"]:
    seen.update(fields.keys())
pd.Series(seen).sort_values(ascending=False).to_frame("containers")

## Save

## What actually explains the dispositions

Two thirds of the pull is not an analyst decision at all, and the fields that say so are
`SA Decision` (a custom field) and `label` (a container field). Run this before drawing
any conclusion from the raw Disposition counts.

In [ ]:
extra = containers["custom_fields"].apply(lambda d: pd.Series({
    "sa_decision": d.get("SA Decision"),
    "sync_status": d.get("Sync Status"),
    "use_case_label": d.get("Use Case Label"),
}))
containers[extra.columns] = extra

print("SA Decision x Disposition, most common pairs\n")
pairs = (containers.assign(disp=containers["disposition"].fillna("(not set)"))
         .groupby(["sa_decision", "disp"], dropna=False).size()
         .sort_values(ascending=False).head(12).to_frame("containers"))
display(pairs)

other = containers[containers["disposition"] == "Other"]
unset = containers[containers["disposition"].isna()]
print(f"'Other' is {len(other):,} containers, of which SA Decision == 'Duplicate': "
      f"{(other['sa_decision'] == 'Duplicate').sum():,} "
      f"({100 * (other['sa_decision'] == 'Duplicate').mean():.1f}%)")
print(f"'not set' is {len(unset):,} containers, of which label == 'heartbeat': "
      f"{(unset['label'] == 'heartbeat').sum():,} "
      f"({100 * (unset['label'] == 'heartbeat').mean():.1f}%)")

In [ ]:
# The population an analyst actually dispositioned: neither a heartbeat nor a duplicate.
real = containers[(containers["label"] != "heartbeat") & (containers["sa_decision"] != "Duplicate")]
vc = real["disposition"].fillna("(not set)").value_counts().to_frame("containers")
vc["%"] = (100 * vc["containers"] / len(real)).round(1)
print(f"{len(real):,} of {len(containers):,} containers, once heartbeats and duplicates are out\n")
display(vc)

print("\nstill unset after that, by label: these are automation, not analyst queues\n")
display(real[real["disposition"].isna()]["label"].fillna("(none)").value_counts().head(10).to_frame("containers"))

## CEF

`GET /rest/artifact` returns `cef` and the `container` it belongs to, and takes the same
time filter as the container list. That makes this one bulk pull per server rather than
83,493 per-container calls, which at ~0.3s each would be about seven hours.

A container can hold several artifacts, so **`CEF` is a list**, one entry per artifact,
in artifact id order. Nothing is merged or flattened away; a container with no artifacts
in the window gets an empty list.

In [ ]:
def fetch_artifact_page(server: dict, page: int) -> dict:
    params = {
        f"_filter_{TIME_FIELD}__gte": f'"{SINCE_ISO}"',
        "page_size": ARTIFACT_PAGE_SIZE,
        "page": page,
        "sort": "id",
        "order": "asc",
    }
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(
                f"{server['base']}/rest/artifact",
                headers={"ph-auth-token": server["token"]},
                params=params,
                verify=VERIFY_TLS,
                timeout=REQUEST_TIMEOUT,
            )
            r.raise_for_status()
            return r.json()
        except Exception as exc:  # noqa: BLE001
            last_error = exc
            if attempt == MAX_RETRIES - 1:
                break
            time.sleep(2 ** attempt)
    raise RuntimeError(f"{server['name']} artifact page {page} failed: {last_error}")


def fetch_cef(server: dict) -> dict[int, list]:
    """{container_id: [cef, ...]} for one server.

    Only `container` and `cef` are kept. The rest of the artifact record is
    dropped as each page arrives: holding whole artifacts for a quarter of a
    million rows is gigabytes for fields nobody asked for.
    """
    first = fetch_artifact_page(server, 0)
    total = first.get("count", 0)
    pages = -(-total // ARTIFACT_PAGE_SIZE)
    print(f"{server['name']}: {total:,} artifacts, {pages} pages", flush=True)

    by_container: dict[int, list] = {}
    seen = 0

    def absorb(batch):
        nonlocal seen
        for art in batch:
            cid = art.get("container")
            if cid is None:
                continue
            by_container.setdefault(cid, []).append(art.get("cef") or {})
            seen += 1

    absorb(first.get("data", []))
    for page in range(1, pages):
        batch = fetch_artifact_page(server, page).get("data", [])
        if not batch:
            print(f"  page {page} empty, stopping early", flush=True)
            break
        absorb(batch)
        if page % 25 == 0 or page == pages - 1:
            print(f"  {seen:,}/{total:,} artifacts, {len(by_container):,} containers", flush=True)
    return by_container


if FETCH_CEF:
    started = time.time()
    cef_by_server = {srv["name"]: fetch_cef(srv) for srv in SERVERS}
    total_arts = sum(len(v) for m in cef_by_server.values() for v in m.values())
    print(f"\n{total_arts:,} artifacts across {sum(len(m) for m in cef_by_server.values()):,} "
          f"containers in {time.time() - started:.0f}s")

In [ ]:
if FETCH_CEF:
    containers["CEF"] = [
        cef_by_server.get(srv, {}).get(cid, [])
        for srv, cid in zip(containers["server"], containers["id"])
    ]
    containers["cef_artifact_count"] = containers["CEF"].apply(len)

    matched = (containers["cef_artifact_count"] > 0).sum()
    print(f"{matched:,} of {len(containers):,} containers have CEF "
          f"({100 * matched / len(containers):.1f}%)")
    # A container whose artifacts predate the window keeps an empty list rather
    # than disappearing: artifact_count is what SOAR says it has, and
    # cef_artifact_count is what this pull actually found, so the two disagreeing
    # is visible instead of silent.
    gap = containers[(containers["artifact_count"] > 0) & (containers["cef_artifact_count"] == 0)]
    print(f"{len(gap):,} containers report artifacts but none landed in the window")
    display(containers.loc[containers["cef_artifact_count"] > 0,
                           ["server", "id", "disposition", "artifact_count", "cef_artifact_count"]].head())

In [ ]:
import gzip

os.makedirs(OUTPUT_DIR, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%d")
base = f"{OUTPUT_DIR}/soar-containers-{MONTHS_BACK}mo-{stamp}"


def jsonable(value):
    """pandas Timestamps and NaT are not JSON; everything else passes through."""
    if value is pd.NaT or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value


if WRITE_JSONL:
    path = f"{base}.jsonl.gz"
    records = containers.to_dict(orient="records")
    with gzip.open(path, "wt", encoding="utf-8") as fh:
        for rec in records:
            fh.write(json.dumps({k: jsonable(v) for k, v in rec.items()}) + "\n")
    print(f"wrote {path}  ({len(records):,} rows, {os.path.getsize(path) / 1e6:.0f} MB)")
    print("  read back with: pd.read_json(path, lines=True)  /  gzip.open + json.loads per line")

if WRITE_CSV:
    flat = containers.copy()
    if not CSV_INCLUDES_CEF:
        flat = flat.drop(columns=[c for c in ("CEF",) if c in flat.columns])
    for col in ("custom_fields", "CEF"):
        if col in flat.columns:
            flat[col] = flat[col].apply(json.dumps)
    suffix = "-with-cef" if ("CEF" in flat.columns) else ""
    path = f"{base}{suffix}.csv"
    flat.to_csv(path, index=False)
    print(f"wrote {path}  ({len(flat):,} rows, {os.path.getsize(path) / 1e6:.0f} MB)")

if WRITE_PARQUET:
    # CEF goes in as a LIST OF JSON STRINGS, one per artifact, NOT as a native
    # struct. Across 256,855 artifacts there are 1,918 distinct CEF keys and
    # 1,614 of them appear on under 1% of artifacts, so a struct schema would be
    # ~1,900 columns that are almost entirely null: slower to write, larger on
    # disk, and miserable to query. A list of strings keeps the per-artifact
    # boundaries and lets the columns people actually filter on stay native.
    pq = containers.copy()

    # A custom field lifted into its own column is NOT reliably a scalar.
    # "Use Case Label" is a string on 81,498 containers and a LIST on 38 of
    # them (e.g. ['malicious_ip', 'malicious_url']). Parquet needs one type per
    # column, so 38 rows in 123,897 were enough to fail the entire write with
    # "Expected bytes, got a 'list' object". Anything non-scalar is JSON
    # encoded here rather than dropped, and the JSONL keeps it native.
    for col in ("sa_decision", "sync_status", "use_case_label", "disposition"):
        if col in pq.columns:
            pq[col] = pq[col].apply(
                lambda v: json.dumps(v) if isinstance(v, (list, dict)) else v
            )

    if "CEF" in pq.columns:
        pq["CEF"] = pq["CEF"].apply(lambda arts: [json.dumps(a) for a in (arts or [])])
    pq["custom_fields"] = pq["custom_fields"].apply(json.dumps)
    try:
        path = f"{base}.parquet"
        pq.to_parquet(path, index=False, compression="zstd")
        print(f"wrote {path}  ({os.path.getsize(path) / 1e6:.0f} MB)")
        print("  columnar: pd.read_parquet(path, columns=[...]) reads 4 columns in ~0.02s")
    except Exception as exc:  # pyarrow / fastparquet not installed
        print(f"parquet skipped: {exc}")

## Single container endpoint reference

What this notebook used to be: each GET in the
[SOAR 8.7.0 container reference](https://help.splunk.com/en/splunk-soar/soar-on-premises/rest-api-reference/8.7.0/container-endpoints/rest-containers),
now pointed at a container from the pull above and the server it actually came from,
instead of an id that had to be pasted in.

In [ ]:
# Pick any row from the pull. Default: the newest container that HAS a disposition,
# so the custom_fields section below is not empty.
with_disp = containers[containers["disposition"].notna()]
sample = (with_disp if len(with_disp) else containers).iloc[-1]

CONTAINER_ID = int(sample["id"])
SERVER = next(s for s in SERVERS if s["name"] == sample["server"])
print(f"{SERVER['name']} container {CONTAINER_ID}: {sample['name']!r}  disposition={sample['disposition']!r}")


def get(path, params=None):
    """GET a SOAR REST path on SERVER, print status and URL, return JSON (or text)."""
    r = requests.get(
        f"{SERVER['base']}{path}",
        headers={"ph-auth-token": SERVER["token"]},
        params=params,
        allow_redirects=False,
        verify=VERIFY_TLS,
        timeout=REQUEST_TIMEOUT,
    )
    print(r.status_code, r.url)
    try:
        return r.json()
    except ValueError:
        return r.text


def show(obj, limit=5000):
    print(json.dumps(obj, indent=2, default=str)[:limit])

### `GET /rest/container/<container_id>`

In [ ]:
container = get(f"/rest/container/{CONTAINER_ID}")
show(container)

In [ ]:
pd.Series(container).to_frame("value") if isinstance(container, dict) else container

### `GET /rest/container/<container_id>?_annotation_authorized_users`
Adds the authorized users field to the container.

In [ ]:
container_annotated = get(f"/rest/container/{CONTAINER_ID}", params="_annotation_authorized_users")
show(container_annotated)

### `GET /rest/container/<container_id>/permitted_users`
Non-administrator users with permission to view the container.

In [ ]:
permitted_users = get(f"/rest/container/{CONTAINER_ID}/permitted_users")
show(permitted_users)

### `GET /rest/container/<container_id>/approvals`

In [ ]:
approvals = get(f"/rest/container/{CONTAINER_ID}/approvals")
show(approvals)

### `GET /rest/container_options`
Not tied to a container: the statuses, severities, tags, sensitivity levels and labels
available on this server.

In [ ]:
container_options = get("/rest/container_options")
show(container_options)